[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/YOUR_REPO/blob/main/session1_weavers_three_cases.ipynb)


# Weaver's Three Kinds of Problems — A Hands-On Tour

Weaver (1948) argued that science had only really mastered two kinds of problems, leaving a huge
middle ground — **organized complexity** — mostly untouched. This notebook builds all three, one
at a time, and instead of just eyeballing a plot and saying "that looks more/less predictable," we
actually **measure** how predictable each one is, using the same simple test throughout:

> Change one small thing (repeat the run exactly, use a different random seed, or nudge the setup
> slightly) — how much does the result move?

Small, clearly-named functions throughout, each with a one-line explanation of what it does —
you should never have to read more than a few lines at once to understand a step.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams["figure.figsize"] = (7, 4.5)
plt.rcParams["axes.grid"] = True

print("Ready.")


---
## Case 1 — Simplicity

**Everyday example:** you throw a ball. Two numbers — launch angle and speed — determine the
entire flight path. No individuals, no randomness, nothing hidden. This is the kind of problem
classical physics mastered first.


In [ ]:
GRAVITY = 9.81       # a constant, doesn't change
LAUNCH_SPEED = 20.0  # also fixed for this whole notebook

def ball_path(angle_degrees):
    """Compute the (x, y) flight path of a ball thrown at `angle_degrees`."""
    angle = np.radians(angle_degrees)
    vx = LAUNCH_SPEED * np.cos(angle)   # horizontal speed
    vy = LAUNCH_SPEED * np.sin(angle)   # vertical speed

    flight_time = 2 * vy / GRAVITY
    t = np.linspace(0, flight_time, 200)   # 200 points in time, from launch to landing

    x = vx * t
    y = vy * t - 0.5 * GRAVITY * t**2
    return x, y


def landing_distance(angle_degrees):
    """How far the ball travels before it lands."""
    x, y = ball_path(angle_degrees)
    return x[-1]   # the last x-value, i.e. where it landed


In [ ]:
plt.figure()
for angle in [20, 35, 45, 60, 75]:
    x, y = ball_path(angle)
    plt.plot(x, y, label=f"{angle}°")
plt.title("A ball thrown at different angles")
plt.xlabel("distance traveled")
plt.ylabel("height")
plt.legend(title="launch angle")
plt.ylim(bottom=0)
plt.show()


### Reproducibility test: run the exact same throw twice

This is the baseline every other case in this notebook gets compared against.


In [ ]:
run_1 = landing_distance(45)
run_2 = landing_distance(45)

case1_stats = pd.DataFrame([{
    "case": "1. Simplicity",
    "what we changed": "nothing -- exact repeat",
    "what we measured": "landing distance",
    "run 1": round(run_1, 6),
    "run 2": round(run_2, 6),
    "difference": round(abs(run_1 - run_2), 6),
}]).set_index("case")

case1_stats


**Takeaway:** the difference is exactly 0. Not "very small" — exactly zero. Same input, same
output, always. That's the signature of simplicity.


---
## Case 2 — Disorganized complexity

**Everyday example:** gas molecules bouncing around a box, or hundreds of people wandering a plaza
with no destination, bumping and changing direction at random. No individual path is predictable.
Let's check whether the **group** is.


In [ ]:
def create_particles(n_particles, box_size, rng):
    """Give each particle a random starting position and a random velocity."""
    positions = rng.uniform(0, box_size, size=(n_particles, 2))
    velocities = rng.normal(0, 1.0, size=(n_particles, 2))
    return positions, velocities


Moving the particles is really two separate ideas: **(1)** slide everyone forward, then
**(2)** fix up anyone who crossed a wall. Splitting these into two small functions makes each
one easy to read on its own.


In [ ]:
def move_particles(positions, velocities, dt):
    """Slide every particle forward by one small time step."""
    return positions + velocities * dt


def bounce_off_walls(positions, velocities, box_size):
    """If a particle crossed a wall, push it back inside and flip its velocity
    on that axis -- like a ball bouncing off a real wall."""
    for dim in range(2):   # dim 0 = x-axis, dim 1 = y-axis
        went_out_of_bounds = (positions[:, dim] < 0) | (positions[:, dim] > box_size)
        velocities[went_out_of_bounds, dim] *= -1
        positions[:, dim] = np.clip(positions[:, dim], 0, box_size)
    return positions, velocities


def step_particles(positions, velocities, box_size, dt):
    """Advance every particle by one time step: move, then bounce off walls."""
    positions = move_particles(positions, velocities, dt)
    positions, velocities = bounce_off_walls(positions, velocities, box_size)
    return positions, velocities


In [ ]:
def run_particle_simulation(seed, n_particles=300, box_size=10.0, n_steps=200, dt=0.05):
    """Run the whole simulation for one random seed and return where everyone
    ended up, plus everyone's final speed."""
    rng = np.random.default_rng(seed)
    positions, velocities = create_particles(n_particles, box_size, rng)

    for _ in range(n_steps):
        positions, velocities = step_particles(positions, velocities, box_size, dt)

    speeds = np.linalg.norm(velocities, axis=1)   # speed = length of the velocity vector
    return positions, speeds


In [ ]:
positions, speeds = run_particle_simulation(seed=7)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(positions[:20, 0], positions[:20, 1], alpha=0.7)
axes[0].set_title("Where 20 sample particles ended up")
axes[0].set_xlim(0, 10); axes[0].set_ylim(0, 10)

axes[1].hist(speeds, bins=30, color="steelblue", edgecolor="white")
axes[1].set_title(f"Speed of all {len(speeds)} particles")
plt.tight_layout()
plt.show()


### Reproducibility test: one individual vs. the group, across 5 different random seeds

We'll track two things for each seed: where one specific particle (#0) ends up, and what the
**average** speed of the whole population is.


In [ ]:
records = []
for seed in [1, 2, 3, 4, 5]:
    positions, speeds = run_particle_simulation(seed=seed)
    records.append({
        "seed": seed,
        "particle #0's final x-position": round(positions[0, 0], 3),
        "average speed, all particles": round(speeds.mean(), 4),
    })

case2_runs = pd.DataFrame(records)
case2_runs


In [ ]:
box_width = 10.0

one_particle_spread_pct = 100 * (
    case2_runs["particle #0's final x-position"].max()
    - case2_runs["particle #0's final x-position"].min()
) / box_width

group_average_spread_pct = 100 * (
    case2_runs["average speed, all particles"].std()
    / case2_runs["average speed, all particles"].mean()
)

print(f"One particle's position swings across {one_particle_spread_pct:.0f}% of the box, run to run.")
print(f"The population's average speed varies by only {group_average_spread_pct:.1f}%, run to run.")


**Takeaway:** you cannot predict where particle #0 will be — it's all over the box depending on
the random seed. But the **average speed of the whole population** barely moves at all between
runs. Individually unpredictable, collectively stable — exactly Weaver's disorganized-complexity
claim, now with a number attached to each half of it.


---
## On to Case 3 — but first, a gap in Weaver's own paper

Weaver named a third category — **organized complexity** — where variables are genuinely
*interrelated*, not just numerous. But he wrote in 1948, and he never specified a mechanism for
it: no equations, no algorithm, nothing you could actually build and run.

What follows is **not** Weaver's own content. It's a mechanism that later complexity science
supplied — **memory and feedback** — to give his abstract category a concrete, runnable form.
We'll say so explicitly whenever we lean on it, so it stays clear what's Weaver's argument and
what's the tool we're borrowing to demonstrate it.


---
## Case 3 — Organized complexity

**Setup:** walkers moving across a grid between three fixed "buildings." Like the NetLogo model
this is based on, walkers **always know exactly where their destination is** — an omniscient
"compass," not something they discover by exploring. We're not modeling that kind of search here.

**What we're actually testing is much narrower:** when a walker has several equally-good steps
available, does it choose based on the past, or not? That single difference is the entire
experiment. Everything else — the world, the walkers, the destinations, which steps are even
allowed — is identical in both models we'll compare.


In [ ]:
WIDTH, HEIGHT = 41, 31
BUILDINGS = [(3, 5), (3, 25), (37, 15)]

print(f"A {WIDTH} x {HEIGHT} grid with {len(BUILDINGS)} buildings: {BUILDINGS}")


### Step 1 — Which moves are even allowed?

A walker only ever considers the up-to-8 neighboring cells that would take it **strictly closer**
to its destination. Two small functions: one to measure distance, one to use that measurement to
filter the 8 neighbors.


In [ ]:
def squared_distance(x1, y1, x2, y2):
    """Squared straight-line distance between two points.
    (We only ever *compare* distances, so there's no need for a slower square root.)"""
    return (x1 - x2) ** 2 + (y1 - y2) ** 2


def closer_neighbors(x, y, goal_x, goal_y):
    """Return every neighboring cell that is strictly closer to (goal_x, goal_y)
    than (x, y) currently is."""
    current_distance = squared_distance(x, y, goal_x, goal_y)

    eight_directions = [
        (1, 0), (-1, 0), (0, 1), (0, -1),
        (1, 1), (1, -1), (-1, 1), (-1, -1),
    ]

    closer_cells = []
    for dx, dy in eight_directions:
        nx, ny = x + dx, y + dy
        if 0 <= nx < WIDTH and 0 <= ny < HEIGHT:
            if squared_distance(nx, ny, goal_x, goal_y) < current_distance:
                closer_cells.append((nx, ny))

    return closer_cells


# quick sanity check: standing at (10, 10), heading to building (37, 15)
print("Example -- cells that get you closer:", closer_neighbors(10, 10, 37, 15))


### Step 2 — The ONE rule that differs between our two models

Usually a walker has 2 or 3 equally "closer" cells to choose from. **How it breaks that tie is
the entire experiment.** We'll write this as three small functions instead of one dense one:


In [ ]:
def choose_randomly(candidates, rng):
    """Pick uniformly at random among the candidate cells."""
    idx = rng.integers(len(candidates))
    return candidates[idx]


def choose_most_popular(candidates, popularity, rng):
    """Pick whichever candidate has been stepped on the most so far
    (ties among equally-popular candidates broken randomly)."""
    popularity_values = [popularity[cy, cx] for cx, cy in candidates]
    best_value = max(popularity_values)
    most_popular = [c for c, v in zip(candidates, popularity_values) if v == best_value]
    idx = rng.integers(len(most_popular))
    return most_popular[idx]


def choose_next_cell(candidates, popularity, use_memory, follow_probability, rng):
    """Model A (use_memory=False): always choose randomly.
    Model B (use_memory=True): usually choose the most-popular candidate,
    but still choose randomly sometimes (with probability `1 - follow_probability`)
    -- so exploration never fully stops."""
    if not use_memory or rng.random() > follow_probability:
        return choose_randomly(candidates, rng)
    return choose_most_popular(candidates, popularity, rng)


### Step 3 — Put it together: walkers shuttling between buildings

Each walker repeatedly walks from one building to another. We'll build this from small pieces:
who starts where, what happens when a walker arrives, and what happens on a single step — then
combine those into the full simulation.


In [ ]:
def make_walkers(n_walkers, rng):
    """Give each walker a random starting building and a different goal building."""
    positions, goals = [], []
    for _ in range(n_walkers):
        start_idx, goal_idx = rng.choice(len(BUILDINGS), size=2, replace=False)
        positions.append(BUILDINGS[start_idx])
        goals.append(BUILDINGS[goal_idx])
    return positions, goals


def pick_new_goal(current_goal, rng):
    """When a walker arrives, send it to a different building."""
    other_buildings = [b for b in BUILDINGS if b != current_goal]
    idx = rng.integers(len(other_buildings))
    return other_buildings[idx]


def take_one_step(x, y, goal_x, goal_y, popularity, use_memory, follow_probability, rng):
    """Move one walker one step closer to its goal, and record the cell it lands on."""
    candidates = closer_neighbors(x, y, goal_x, goal_y)
    next_cell = choose_next_cell(candidates, popularity, use_memory, follow_probability, rng)
    popularity[next_cell[1], next_cell[0]] += 1
    return next_cell


In [ ]:
def run_simulation(use_memory, seed, n_walkers=80, total_steps=20000, follow_probability=0.85):
    """Simulate `n_walkers` shuttling between buildings for a combined `total_steps`
    footsteps. Returns a HEIGHT x WIDTH grid counting how many times each cell was
    stepped on."""
    rng = np.random.default_rng(seed)
    popularity = np.zeros((HEIGHT, WIDTH), dtype=int)
    positions, goals = make_walkers(n_walkers, rng)

    steps_taken = 0
    while steps_taken < total_steps:
        for i in range(n_walkers):
            x, y = positions[i]
            goal_x, goal_y = goals[i]

            if (x, y) == (goal_x, goal_y):   # arrived: head somewhere new
                goals[i] = pick_new_goal(goals[i], rng)
                continue

            positions[i] = take_one_step(
                x, y, goal_x, goal_y, popularity, use_memory, follow_probability, rng
            )
            steps_taken += 1
            if steps_taken >= total_steps:
                break

    return popularity

print("Simulation function ready.")


### Step 4 — Run both models once, and look at them

Same seed, same walker count, same step budget. The only thing that changes is `use_memory`.

**Model A is the control for this experiment** — not a second instance of Case 2's disorganized
complexity. These walkers are goal-directed the whole time; they're just uncoordinated with each
other.


In [ ]:
model_A = run_simulation(use_memory=False, seed=123)   # control: no memory
model_B = run_simulation(use_memory=True, seed=123)    # treatment: memory + feedback

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, popularity, title in [
    (axes[0], model_A, "Model A -- No memory (control)"),
    (axes[1], model_B, "Model B -- Memory + feedback"),
]:
    ax.imshow(popularity, origin="lower", cmap="inferno")
    bx, by = zip(*BUILDINGS)
    ax.scatter(bx, by, marker="s", s=80, c="cyan", edgecolor="white")
    ax.set_title(title)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
plt.suptitle("Cumulative foot traffic (brighter = more steps taken on that cell)")
plt.tight_layout()
plt.show()


### Step 5 — Summary statistics for the two models

A picture tells you *that* something is different. These numbers tell you **how**:

- **cells ever used** — how much of the map saw any traffic at all.
- **busiest cell's traffic** — the single most-walked-on cell's footstep count.
- **top-5% share** — of all footsteps taken, what fraction landed in the busiest 5% of used cells?
  Higher = more concentrated onto a few preferred routes.
- **traffic entropy** — a single number measuring how spread out the traffic is (higher = more
  spread out). Same idea as top-5% share, computed a different way, as a cross-check.


In [ ]:
def summarize(popularity, model_name):
    """Compute a few simple numbers describing how concentrated the foot traffic became."""
    used_cells = popularity[popularity > 0]
    total_footsteps = used_cells.sum()

    sorted_traffic = np.sort(used_cells)[::-1]
    n_top = max(1, int(len(sorted_traffic) * 0.05))
    top5_share = sorted_traffic[:n_top].sum() / total_footsteps

    p = used_cells / total_footsteps
    traffic_entropy = -(p * np.log(p)).sum()

    return {
        "model": model_name,
        "total footsteps": int(total_footsteps),
        "cells ever used": int(len(used_cells)),
        "busiest cell's traffic": int(used_cells.max()),
        "top-5% share": round(top5_share, 3),
        "traffic entropy": round(traffic_entropy, 3),
    }

summary_table = pd.DataFrame([
    summarize(model_A, "A: No memory"),
    summarize(model_B, "B: Memory + feedback"),
]).set_index("model")

summary_table


**What to read off this table:** Model B concentrates a noticeably larger share of all footsteps
onto a small set of cells, and has lower traffic entropy — both numbers agree that memory makes
movement more organized around a few preferred routes, even though it doesn't use noticeably more
or fewer cells overall.


### Step 6 — The real test: does this repeat?

One run each isn't enough to know *what kind* of difference this is. Let's run both models 5
times, changing only the random seed, and see how much the results actually move around.


In [ ]:
seeds = [1, 2, 3, 4, 5]

records = []
for model_name, use_memory in [("A: No memory", False), ("B: Memory + feedback", True)]:
    for seed in seeds:
        popularity = run_simulation(use_memory=use_memory, seed=seed)
        stats = summarize(popularity, model_name)
        stats["seed"] = seed
        records.append(stats)

repeated_runs = pd.DataFrame(records)
repeated_runs[["model", "seed", "top-5% share", "traffic entropy"]]


In [ ]:
reproducibility_summary = repeated_runs.groupby("model")[["top-5% share", "traffic entropy"]].agg(
    ["mean", "std", "min", "max"]
)
reproducibility_summary


**What to read off this table:** compare the `std` (standard deviation) columns between the two
models. Model A's numbers barely move between runs — a small, tight `std`. Model B's numbers move
noticeably more, run to run, even though nothing about the rules or the setup changed — only which
random walker happened to be where, early on.

That's the signature of organized complexity that **memory and feedback** — the mechanism we
borrowed, not Weaver's own text — produce: not that the group behaves in a more complicated way,
but that the group-level *outcome itself* stops being reliably predictable from the parameters
alone. No amount of knowing `n_walkers`, `follow_probability`, or the building layout in advance
tells you exactly how concentrated Model B's traffic will end up being — you have to run it and
see.


---
## Recap: all three of Weaver's cases, side by side

| | Case 1: Simplicity | Case 2: Disorganized | Case 3: Organized (Model A vs. B) |
|---|---|---|---|
| how many things involved | 2–3 | hundreds, independent | dozens, interdependent (Model B only) |
| individually predictable? | yes, exactly | no | yes -- each walker follows its own rule |
| what varies between repeated runs | nothing (0% by construction) | individual paths, not the group average | Model B's *group-level* outcome itself |
| reproducible across seeds? | trivially, yes | yes, tightly, at the group level | Model A: yes. Model B: noticeably not |

**The key move in Case 3:** Model A already shows you what "goal-directed but uncoordinated"
looks like — it is *not* a repeat of Case 2. The organized-complexity signature only shows up once
memory and feedback are added in Model B, and even then, it's borrowed machinery Weaver's own 1948
paper never supplied.

**Next up:** an agent-based model that builds on this Case 3 mechanism directly — this is where
the course's core modeling toolkit begins.
